# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR<sup>2</sup> mass spectrometry dataset using the `mlcroissant` library, referencing dataset entities by their `@id` fields throughout.

### Dataset Source
Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

*All record sets, fields, and columns will be referenced using their Croissant `@id` values, as per best practices for schema-based interoperability.*

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the Croissant FAIR² mass spectrometry dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Set the Croissant metadata URL for the FAIR^2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a dataclass, not a dict

# Print dataset name and description
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)


## 2. Data Overview
Review all available record sets present in the dataset and their properties (using their `@id`).

The record sets define the tables available for analysis. For each record set, we also list available field `@id` values (which map to data columns).

In [ ]:
record_sets = dataset.record_sets

print(f"Available record sets (table schemas) in dataset:\n")
for rset in record_sets:
    print(f"- RecordSet name: {rset.name}")
    print(f"  @id: {rset.id}")
    # Print associated fields/columns
    field_ids = [field.id for field in rset.fields]
    print(f"  Field @ids: {field_ids}\n")

Let's print a small sample from the first record set using its `@id`. You can replace this with any other valid record set `@id` obtained above.

In [ ]:
# Example: examine sample records from the first record set by @id
# (Replace as needed with your desired record set @id)
first_record_set_id = record_sets[0].id
print(f"Sample records for record set @id: {first_record_set_id}")
for i, row in enumerate(dataset.records(record_set=first_record_set_id)):
    print(row)
    if i == 2:
        break

## 3. Data Extraction
Load data from all major record sets into pandas DataFrames using the record set `@id`. Each field and column is accessible by its Croissant `@id` as shown above.

Let's load all available record sets as DataFrames for downstream analysis.

In [ ]:
# Load all record sets as pandas DataFrames, mapping by their @id
dataframes = {}

for rset in dataset.record_sets:
    rset_id = rset.id
    records = list(dataset.records(record_set=rset_id))
    df = pd.DataFrame(records)
    dataframes[rset_id] = df
    print(f"Loaded {len(df)} records for record set \"{rset.name}\" (@id: {rset_id}) with columns:")
    print(list(df.columns))

# Pick one main table for exploration (usually the primary one with biological data)
main_table_id = record_sets[0].id
print(f"\nPrimary data table chosen for EDA is: {main_table_id}\n")
dataframes[main_table_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's perform some common processing steps. We will select a numeric field (column) for more detailed examination, then filter, normalize, and group as examples.

*Fields, columns, and groupings are referenced by their `@id` values. Replace with desired columns from above if needed.*

In [ ]:
# Identify a suitable numeric column by @id. We'll display several column candidates first.
df = dataframes[main_table_id]
print(f"Columns in primary table ({main_table_id}):\n{list(df.columns)}\n")

# For demonstration, let's use 'cr:peptide.counts' (peptide count), a typical numeric field.
# If not available, use any available numeric field @id, e.g., 'cr:coverage.percent', 'cr:MW', etc.

numeric_field_id = None
numeric_field_candidates = ['cr:peptide.counts', 'cr:coverage.percent', 'cr:MW', 'cr:pI', 'cr:abundance.A', 'cr:abundance.B']
for candidate in numeric_field_candidates:
    if candidate in df.columns:
        numeric_field_id = candidate
        print(f"Using numeric field for analysis: {candidate}")
        break
    
if numeric_field_id is None:
    print("No suitable numeric field found. Please check available columns and update the candidates list.")

# Remove rows with missing values in numeric_field_id
filtered_df = df[df[numeric_field_id].notnull()]
threshold = filtered_df[numeric_field_id].quantile(0.75)  # Use 75th percentile as example threshold
filtered_df = filtered_df[filtered_df[numeric_field_id].astype(float) > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
print(filtered_df.head())

# Normalize the selected numeric field (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id].astype(float) - filtered_df[numeric_field_id].astype(float).mean()
) / filtered_df[numeric_field_id].astype(float).std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Select a categorical/grouping field: e.g., by 'cr:sample' (or similar @id in the columns if present)
group_field_id = None
group_candidates = ['cr:sample', 'cr:modification', 'cr:accession', 'cr:description']
for candidate in group_candidates:
    if candidate in filtered_df.columns:
        group_field_id = candidate
        print(f"Using group field for aggregation: {candidate}")
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index(name=f"{numeric_field_id}_mean")
    print(f"\nGrouped data by {group_field_id}, showing mean {numeric_field_id} per group:")
    print(grouped_df.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and, if grouped, compare means between groups.

*All axes and legends reference Croissant `@id` values for consistency and reproducibility.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id].astype(float), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If grouping field exists, plot barplot of means
if group_field_id and 'grouped_df' in locals():
    # Show top 10 groups by mean (if many)
    plt.figure(figsize=(10,4))
    top_n = grouped_df.sort_values(f"{numeric_field_id}_mean", ascending=False).head(10)
    sns.barplot(x=group_field_id, y=f"{numeric_field_id}_mean", data=top_n)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion

In this notebook, we used the `mlcroissant` library for robust and schema-informed access to the FAIR<sup>2</sup> mass spectrometry dataset. All dataset entities (record sets, columns, fields) were referenced by Croissant `@id`, ensuring clarity and reproducibility of any downstream analyses.

Key findings and next steps:
- The dataset contains detailed protein-level information, with numeric fields such as peptide counts and abundance values.
- Data was loaded, filtered, normalized, and grouped by categorical variables, all by schema-conformant keys.
- Distributions and group means can quickly be visualized for biological interpretation or downstream machine learning.

*For further analysis, continue using the field and record set `@id`s from the schema to ensure your transformations are directly tied to the FAIR data standard.*